# 🎓 Training Pipeline YOLOv8 & CrossKD (RDD2022)
### Tugas Akhir: *Pengembangan Model Lightweight YOLOv8 untuk Deteksi Kerusakan Jalan Menggunakan Cross-Head Knowledge Distillation*
**Peneliti:** Iqbal Fahrozi (11221034)  
**Dosen Pembimbing:** Bima Prihasto, S.Si., M.Si., Ph.D. & Boby Mugi Pratama, S.Si., M.Han.

---
### 📌 Panduan Singkat:
1. Pastikan **Runtime GPU** sudah aktif: menu **Runtime** > **Change runtime type** > pilih **T4 GPU** > Save.
2. Jalankan sel secara berurutan dari atas ke bawah (**Shift + Enter**).
3. Checkpoint model otomatis disimpan langsung ke Google Drive kamu (`TA CROSSKD/runs/detect`), sehingga aman jika sesi terputus.

## 1. Cek Ketersediaan GPU

In [ ]:
!nvidia-smi

import torch
print("=" * 50)
print(f"CUDA Tersedia   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Terdeteksi  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Tersedia   : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 50)

## 2. Hubungkan Google Drive (Mount Drive)
File `code_lkd_fix.zip` dan `dataset_rdd2022_augmented.zip` akan dibaca dari Google Drive kamu.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Hubungkan ke Google Drive
drive.mount('/content/drive')

# Deteksi otomatis lokasi folder TA CROSSKD di Google Drive
possible_paths = [
    Path("/content/drive/MyDrive/Iqbal/TA CROSSKD"),
    Path("/content/drive/MyDrive/TA CROSSKD"),
    Path("/content/drive/Shareddrives/TA CROSSKD"),
]

DRIVE_DIR = None
for p in possible_paths:
    if p.exists():
        DRIVE_DIR = p
        break

if DRIVE_DIR:
    print(f"[✓] Folder Google Drive ditemukan: {DRIVE_DIR}")
    print("Isi folder:", [f.name for f in DRIVE_DIR.iterdir()])
else:
    print("[!] Peringatan: Folder 'TA CROSSKD' belum ditemukan otomatis.")
    print("Silakan cek path Google Drive kamu di panel sebelah kiri.")

## 3. Ekstrak Kode & Dataset ke Disk Lokal Colab (`/content/`)
> **Catatan:** Membaca puluhan ribu gambar dari disk lokal Colab **10x lebih cepat** daripada membaca langsung dari Google Drive yang di-mount.

In [ ]:
import os
from pathlib import Path

# 1. Ekstrak Kode Program
code_zip = DRIVE_DIR / "code_lkd_fix.zip"
if not Path("/content/ultralytics").exists() and code_zip.exists():
    print(">>> Mengekstrak code_lkd_fix.zip ke /content/ ...")
    !unzip -q "{code_zip}" -d /content/
    print("[✓] Kode berhasil diekstrak!")
else:
    print("[✓] Kode sudah siap di /content/")

# 2. Ekstrak Dataset (13.4 GB)
dataset_zip = DRIVE_DIR / "dataset_rdd2022_augmented.zip"
if not Path("/content/dataset_rdd2022_augmented/images").exists() and dataset_zip.exists():
    print(">>> Mengekstrak dataset_rdd2022_augmented.zip (butuh waktu ~1-2 menit di NVMe Colab)... ")
    !unzip -q "{dataset_zip}" -d /content/
    print("[✓] Dataset berhasil diekstrak!")
else:
    print("[✓] Dataset sudah siap di /content/dataset_rdd2022_augmented/")

## 4. Sesuaikan Path `data.yaml` & Install Dependensi

In [ ]:
# 1. Hapus wandb agar tidak menginterupsi training dengan error nama project
!pip uninstall -y -q wandb

# 2. Install dependensi pendukung YOLOv8
!pip install -q opencv-python-headless matplotlib pyyaml scipy tqdm psutil pandas seaborn thop

# 3. Perbarui path di data.yaml agar mengarah ke disk lokal Colab (/content/)
yaml_path = Path("/content/dataset_rdd2022_augmented/data.yaml")
if yaml_path.exists():
    with open(yaml_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Ganti path root dataset ke /content/dataset_rdd2022_augmented
    import re
    content = re.sub(r"path:.*", "path: /content/dataset_rdd2022_augmented", content)
    
    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("[✓] data.yaml berhasil diperbarui!")
    print(content)
else:
    print("[!] data.yaml tidak ditemukan di path yang diharapkan.")

## 5. 🚀 Mulai Training Eksperimen 2: Teacher YOLOv8m (Optimized Anti-Overfitting)
### Fitur Optimasi Sesuai Arahan Pak Boby:
* **`patience: 20`** &rarr; Early Stopping sejati (stop otomatis jika 20 epoch tidak ada kenaikan mAP).
* **`cos_lr: True`** &rarr; Cosine Annealing Learning Rate scheduler agar penurunan LR lebih halus.
* **`close_mosaic: 15`** &rarr; Mematikan augmentasi mosaic di 15 epoch terakhir.
* **`weight_decay: 0.001`** &rarr; Regularisasi L2 ditingkatkan untuk mengerem overfitting.
* **`Symlink Google Drive`** &rarr; **Semua checkpoint `last.pt` & `best.pt` otomatis tersimpan langsung ke Google Drive** sehingga aman dari resiko session disconnect!

In [ ]:
import os
import sys

# 1. Matikan integrasi wandb secara permanen
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

# 2. Masukkan direktori kerja ke sys.path
sys.path.insert(0, "/content")
os.chdir("/content")

# 3. Buat folder dan symlink ke Google Drive
# Folder 'TA_PROJECT' tanpa tanda garis miring (/) agar 100% bebas dari batasan karakter nama project
if DRIVE_DIR:
    drive_detect = DRIVE_DIR / "runs" / "detect"
    drive_detect.mkdir(parents=True, exist_ok=True)
    !rm -rf /content/TA_PROJECT
    !ln -s "{drive_detect}" /content/TA_PROJECT
    print(f"[✓] Symlink aktif: /content/TA_PROJECT -> {drive_detect}")

from ultralytics import YOLO, settings
settings.update({"wandb": False})

# Kosongkan callback wandb di memory jika ada
try:
    import ultralytics.utils.callbacks.wb as wb_cb
    wb_cb.callbacks.clear()
except Exception:
    pass

print("=" * 65)
print(">>> MEMULAI TRAINING TEACHER OPTIMIZED (YOLOv8m) <<<")
print(f"Dataset Path       : /content/dataset_rdd2022_augmented/data.yaml")
print(f"Simpan Checkpoint  : {DRIVE_DIR}/runs/detect/teacher_yolov8m_optimized")
print("Epochs             : 100")
print("Batch Size         : 16")
print("Patience           : 20 (Early Stopping)")
print("Cosine LR          : True")
print("Close Mosaic       : 15")
print("Weight Decay       : 0.001 (L2 Regularization)")
print("AMP (FP16)         : True")
print("=" * 65 + "\n")

# Load bobot pretrained official YOLOv8m
model = YOLO("/content/PretrainedModelFromOfficial/yolov8m.pt")

# Eksekusi Training
# project='TA_PROJECT' tidak mengandung karakter '/', dan otomatis tembus ke Google Drive via symlink!
model.train(
    data="/content/dataset_rdd2022_augmented/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    workers=2,              # 2 worker optimal untuk 2 vCPU Google Colab
    device=0,
    pretrained=True,
    iou=0.6,
    patience=20,            # Early stopping jika 20 epoch tidak membaik
    cos_lr=True,            # Cosine learning rate scheduler
    close_mosaic=15,        # Matikan mosaic di 15 epoch terakhir
    weight_decay=0.001,     # Regularisasi diperkuat
    project="TA_PROJECT",   # Nama bersih tanpa slash '/', otomatis tersimpan di Google Drive
    name="teacher_yolov8m_optimized",
    exist_ok=True,
    Mode="Training",
    quant=False,
    amp=True,
)

print("\n[✓] Training Teacher YOLOv8m Optimized Selesai!")
print(f"File bobot terbaik tersimpan di: {DRIVE_DIR}/runs/detect/teacher_yolov8m_optimized/weights/best.pt")

## 6. 🔄 Sel Darurat: Lanjutkan Training Jika Terputus (Resume Training)
Jika sesi Google Colab kamu mendadak terputus di tengah jalan, jalankan sel di bawah ini untuk **melanjutkan secara otomatis dari epoch terakhir** tanpa mengulang dari awal:

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Path ke checkpoint last.pt di Google Drive
last_ckpt = DRIVE_DIR / "runs" / "detect" / "teacher_yolov8m_optimized" / "weights" / "last.pt"

if last_ckpt.exists():
    print(f">>> Melanjutkan training dari checkpoint: {last_ckpt}")
    model_resume = YOLO(str(last_ckpt))
    model_resume.train(resume=True)
else:
    print(f"[!] Checkpoint '{last_ckpt}' belum ditemukan.")

## 7. 📊 Tampilkan Hasil & Grafik Metrik Evaluasi
Jalankan sel ini setelah training selesai untuk melihat kurva loss, mAP, dan confusion matrix:

In [ ]:
from IPython.display import Image, display
from pathlib import Path

res_dir = DRIVE_DIR / "runs" / "detect" / "teacher_yolov8m_optimized"

plots = ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]
for p in plots:
    img_path = res_dir / p
    if img_path.exists():
        print(f"=== {p} ===")
        display(Image(filename=str(img_path)))
    else:
        print(f"File {p} belum tersedia.")